# Computer Exercise 15.28 — Problem 3

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.28 Sequential Decision Making — *True Rehabilitation of +CNRT via Soft-Q Averaging and Batch Replay*
> **풀이 일자**: Day 95
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 3.** Day 94 (§15.27 Problem 3) attempted to rehabilitate the +CNRT stack's
> cross-slip deficit using prioritized replay $\alpha \in \{0, 0.5, 1\}$ and Noisy-head
> higher-lr multiplier $k \in \{1, 2, 5\}$. The best cell was $(\alpha=0, k=1)$ — i.e.,
> **both prescriptions turned off** — which merely *tied* baseline (cross-slip mean 0.919 vs
> 0.919). No cell in the 9-cell grid actually **exceeded** baseline ($\Delta_{\max} = 0$).
> The prescriptions themselves were often destabilizing ($k=5$ catastrophic almost everywhere).
> **Test whether two different prescriptions — Polyak-averaged target head ("soft-Q
> averaging") and mini-batch replay — succeed where prioritized replay + lr scaling failed.**
> Run a 2×2 factorial: (target-Polyak $\tau_p \in \{0, 0.01\}$) × (batch size $B \in \{1, 16\}$)
> on +CNRT × 3 seeds × 400 steps, freeze, then greedy-deploy under
> $p_d \in \{0.05, 0.10, 0.20\}$. Report cross-slip mean and the max positive gap
> $\Delta_{\max} = \max_{(\tau_p, B)}[R^{+\text{CNRT}}_{\text{avg}} - R^{\text{base}}_{\text{avg}}]$.

### 한국어 풀이용 정리
Day 94 P3 의 (α, k) 처방은 +CNRT 열위를 baseline-tie 로만 회복시켰다. **다른 두 처방** —
target 헤드 Polyak-averaging 과 mini-batch replay — 이 진짜 rehabilitation (baseline 초과)
을 낼 수 있는지 2×2 factorial 로 판정. 각 셀에서 3 슬립 × 3 시드 학습·평가.


## 2. 수학적 배경

### 2.1 Soft-Q averaging (Polyak target)
Target Q-head 파라미터 $\theta'$ 를 훈련 파라미터 $\theta$ 로부터 지수 이동평균으로 갱신:
$$
  \theta' \leftarrow (1-\tau_p) \theta' + \tau_p \, \theta.
$$
Bellman target 계산에 $\theta'$ 를 쓰면 target drift 가 완만해져 categorical head 의
projection error 가 안정화된다.

### 2.2 Mini-batch replay
샘플 $(s_i, a_i, r_i, s_i')$ 를 크기 $M$ FIFO 버퍼에 저장, 갱신 시 균등 확률로
$B$ 개 추출해 gradient 를 평균. Day 94 P3 는 prioritized 였음; 여기서는 균등 (uniform).

### 2.3 지표
Cross-slip mean $\bar R_c = \frac1{|P|}\sum_{p_d} R_c(p_d)$, generalization gap
$\Gamma_c = \max_{p_d} R_c(p_d) - \min_{p_d} R_c(p_d)$, 그리고 rehabilitation gap
$$\boxed{\;\Delta_{\max} = \max_{(\tau_p, B)}\bigl[\bar R^{+\text{CNRT}}(\tau_p, B) - \bar R^{\text{base}}(\tau_p, B)\bigr]. \;}$$


## 3. 풀이 흐름

1. 5-state chain MDP 를 $p_{\text{train}}=0.10$ 로 학습.
2. Learner 확장: `target_W2`, `target_b2` + Polyak update. Replay buffer 추가.
3. baseline (MSE 헤드, no components) 및 +CNRT (Day94-cell 재조립 스택) 두 클래스.
4. 2×2 factorial: $\tau_p \in \{0, 0.01\}$ × $B \in \{1, 16\}$ × condition ∈ {base, +CNRT}
   × 3 seeds (95301–95303) × 400 step.
5. freeze 후 greedy 로 3 slip ($p_d \in \{0.05, 0.10, 0.20\}$) 각 60 에피소드.
6. 셀별 cross-slip mean, gap $\Gamma$, $\Delta = R^{+\text{CNRT}} - R^{\text{base}}$.
7. 표 + heatmap.


In [1]:

import os, sys
sys.path.insert(0, '/tmp/pypkg')
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'
os.environ['HOME'] = '/tmp/home'
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

# 5-state chain MDP (same environment used across Days 90-94)
NS, NA = 5, 2  # 5 states, 2 actions (left/right)
GAMMA = 0.9

def step(s, a, p_slip, rng):
    "Return (s_next, reward)."
    # a=1 means "right", a=0 means "left"; with prob p_slip do opposite
    if rng.random() < p_slip:
        a = 1 - a
    if a == 1:
        s2 = min(NS - 1, s + 1)
    else:
        s2 = max(0, s - 1)
    # reward: +1 at rightmost terminal-like state; -0.1 at leftmost; else 0
    if s2 == NS - 1:
        r = 1.0
    elif s2 == 0:
        r = -0.1
    else:
        r = 0.0
    return s2, r

def phi_state(s):
    v = np.zeros(NS); v[s] = 1.0
    return v

def rollout(policy_fn, p_slip, n_episodes, horizon, seed):
    rng = np.random.default_rng(seed)
    returns = np.zeros(n_episodes)
    for ep in range(n_episodes):
        s = NS // 2
        G, disc = 0.0, 1.0
        for _ in range(horizon):
            a = policy_fn(s, rng)
            s, r = step(s, a, p_slip, rng)
            G += disc * r
            disc *= GAMMA
        returns[ep] = G
    return returns

from collections import deque

class RehabLearner:
    "Categorical Q-learner with optional Polyak-target + batch replay + +CNRT components."
    def __init__(self, seed, use_components=False, tau_polyak=0.0, batch=1,
                 sigma0=0.3, eta=0.05):
        rng = np.random.default_rng(seed)
        self.rng = rng
        self.use = use_components
        self.tp = tau_polyak
        self.B = batch
        self.eta = eta
        self.K = 10
        self.atoms = np.linspace(-1, 1, self.K)
        self.dz = self.atoms[1] - self.atoms[0]
        H = 16; self.H = H
        # Component-specific inits when use_components (Day 94 P1 optima):
        # Noisy sigma_head_scale 0.1, Cramer head 0.3, EMA 0.3, Twin 0.1 -> use shared 0.3
        s = 0.3 if use_components else sigma0
        self.W1 = rng.normal(0, s, (H, NS)); self.b1 = np.zeros(H)
        self.W2 = rng.normal(0, s, (NA, self.K, H)); self.b2 = np.zeros((NA, self.K))
        self.target_W2 = self.W2.copy(); self.target_b2 = self.b2.copy()
        self.target_W1 = self.W1.copy(); self.target_b1 = self.b1.copy()
        if use_components:
            self.noisy_sigma = np.abs(rng.normal(0, 0.03, (NA, self.K, H)))
            self.mu_ema = np.zeros(H); self.var_ema = np.ones(H); self.beta = 0.95
        self.buf = deque(maxlen=1024)
        self.step_no = 0

    def _phi(self, s):
        v = np.zeros(NS); v[s] = 1.0; return v

    def _forward(self, s, target=False):
        x = self._phi(s)
        if target:
            W1, b1, W2, b2 = self.target_W1, self.target_b1, self.target_W2, self.target_b2
        else:
            W1, b1, W2, b2 = self.W1, self.b1, self.W2, self.b2
        h = np.tanh(W1 @ x + b1)
        if self.use and self.step_no > 5 and not target:
            h = (h - self.mu_ema) / (np.sqrt(self.var_ema) + 1e-3)
        if self.use and not target:
            eps = self.rng.normal(0, 1, W2.shape)
            W2e = W2 + self.noisy_sigma * eps
        else:
            W2e = W2
        logits = np.einsum('akh,h->ak', W2e, h) + b2
        m = logits - logits.max(axis=1, keepdims=True)
        p = np.exp(m); p /= p.sum(axis=1, keepdims=True)
        q = p @ self.atoms
        return h, logits, p, q, x

    def act(self, s, eps=0.20):
        if self.rng.random() < eps: return int(self.rng.integers(NA))
        _, _, _, q, _ = self._forward(s); return int(np.argmax(q))

    def _project(self, val):
        val = np.clip(val, -1, 1)
        b = (val - (-1.0)) / self.dz
        lo = int(np.clip(np.floor(b), 0, self.K-1))
        hi = int(np.clip(np.ceil(b), 0, self.K-1))
        m = np.zeros(self.K)
        if lo == hi: m[lo] = 1
        else: m[lo] = hi - b; m[hi] = b - lo
        return m

    def observe(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def learn(self):
        if len(self.buf) == 0: return
        idxs = self.rng.integers(0, len(self.buf), size=self.B)
        grads_W2 = np.zeros_like(self.W2); grads_b2 = np.zeros_like(self.b2)
        grads_W1 = np.zeros_like(self.W1); grads_b1 = np.zeros_like(self.b1)
        for i in idxs:
            s, a, r, s2, done = list(self.buf)[i]
            h, logits, p, q, x = self._forward(s)
            if done:
                m = self._project(r)
            else:
                _, _, _, q2, _ = self._forward(s2, target=(self.tp > 0))
                m = self._project(r + GAMMA * q2[int(np.argmax(q2))])
            pa = p[a]
            if self.use:
                # Cramér (heavier gradient) — component stack
                cdf_p = np.cumsum(pa); cdf_m = np.cumsum(m)
                grad_cdf = 2.0 * (cdf_p - cdf_m)
                grad_p = np.cumsum(grad_cdf[::-1])[::-1]
                grad_l = pa * (grad_p - (pa*grad_p).sum())
            else:
                grad_l = pa - m
            grads_W2[a] += np.outer(grad_l, h) / self.B
            grads_b2[a] += grad_l / self.B
            dh = self.W2[a].T @ grad_l
            dz = dh * (1 - h**2)
            grads_W1 += np.outer(dz, x) / self.B
            grads_b1 += dz / self.B
            if self.use:
                b = self.beta
                self.mu_ema = b * self.mu_ema + (1-b) * h
                self.var_ema = b * self.var_ema + (1-b) * (h - self.mu_ema)**2
        self.W2 -= self.eta * grads_W2; self.b2 -= self.eta * grads_b2
        self.W1 -= self.eta * grads_W1; self.b1 -= self.eta * grads_b1
        if self.tp > 0:
            self.target_W1 = (1-self.tp)*self.target_W1 + self.tp*self.W1
            self.target_b1 = (1-self.tp)*self.target_b1 + self.tp*self.b1
            self.target_W2 = (1-self.tp)*self.target_W2 + self.tp*self.W2
            self.target_b2 = (1-self.tp)*self.target_b2 + self.tp*self.b2
        self.step_no += 1

def train_and_deploy(seed, use_components, tp, B, T=400, p_train=0.10,
                     slips=(0.05,0.10,0.20), n_eval=60):
    lnr = RehabLearner(seed, use_components=use_components, tau_polyak=tp, batch=B)
    rng = np.random.default_rng(seed + 3000)
    s = NS // 2
    for _ in range(T):
        a = lnr.act(s, eps=0.20)
        s2, r = step(s, a, p_train, rng)
        done = (s2 == NS-1) or (s2 == 0)
        lnr.observe(s, a, r, s2, done)
        lnr.learn()
        s = s2 if not done else NS // 2
    # freeze; greedy deployment across slips
    def policy(state, rng_local):
        _, _, _, q, _ = lnr._forward(state); return int(np.argmax(q))
    results = {}
    for pd_ in slips:
        Gs = rollout(policy, pd_, n_eval, 20, seed + 7000 + int(pd_*100))
        results[pd_] = float(np.mean(Gs[-8:]))
    return results


In [2]:

# ---- Run 2x2 factorial x 2 conditions x 3 seeds ----
tps = [0.0, 0.01]
Bs  = [1, 16]
conditions = [('baseline', False), ('+CNRT', True)]
seeds = [95301, 95302, 95303]
slips = (0.05, 0.10, 0.20)

rows = []
for tp in tps:
    for B in Bs:
        for name, use in conditions:
            per_slip = {p: [] for p in slips}
            for sd in seeds:
                res = train_and_deploy(sd, use, tp, B)
                for p, v in res.items():
                    per_slip[p].append(v)
            row = {'tau_p': tp, 'B': B, 'cond': name}
            for p in slips: row[f'R_p={p}'] = np.mean(per_slip[p])
            xs_mean = np.mean([np.mean(per_slip[p]) for p in slips])
            gamma = max(np.mean(per_slip[p]) for p in slips) - min(np.mean(per_slip[p]) for p in slips)
            row['xs_mean'] = xs_mean; row['Gamma'] = gamma
            rows.append(row)

df = pd.DataFrame(rows)
df


,tau_p,B,cond,R_p=0.05,R_p=0.1,R_p=0.2,xs_mean,Gamma
0,0.0000,1,baseline,5.2563,4.7788,4.6037,4.8796,0.6526
1,0.0000,1,+CNRT,4.1054,3.8566,4.2196,4.0606,0.3630
2,0.0000,16,baseline,5.2563,4.7788,4.6037,4.8796,0.6526
3,0.0000,16,+CNRT,4.8995,4.7485,4.4017,4.6832,0.4978
4,0.0100,1,baseline,5.2563,4.7788,4.6037,4.8796,0.6526
5,0.0100,1,+CNRT,4.9438,4.9719,4.5357,4.8171,0.4362
6,0.0100,16,baseline,5.2563,4.7788,4.6037,4.8796,0.6526
7,0.0100,16,+CNRT,4.1488,3.9878,4.0835,4.0733,0.1610


In [3]:

# ---- Cell-wise delta = +CNRT - baseline ----
delta_rows = []
for tp in tps:
    for B in Bs:
        rb = df[(df.tau_p==tp)&(df.B==B)&(df.cond=='baseline')].iloc[0]
        rc = df[(df.tau_p==tp)&(df.B==B)&(df.cond=='+CNRT')].iloc[0]
        delta_rows.append({
            'tau_p': tp, 'B': B,
            'R_base': rb.xs_mean, 'R_CNRT': rc.xs_mean,
            'Delta = R_CNRT - R_base': rc.xs_mean - rb.xs_mean,
            'Gamma_base': rb.Gamma, 'Gamma_CNRT': rc.Gamma,
        })
dfd = pd.DataFrame(delta_rows)
dfd


,tau_p,B,R_base,R_CNRT,Delta = R_CNRT - R_base,Gamma_base,Gamma_CNRT
0,0.0000,1,4.8796,4.0606,-0.8190,0.6526,0.3630
1,0.0000,16,4.8796,4.6832,-0.1963,0.6526,0.4978
2,0.0100,1,4.8796,4.8171,-0.0625,0.6526,0.4362
3,0.0100,16,4.8796,4.0733,-0.8063,0.6526,0.1610


In [4]:

# ---- Heatmap of Delta over (tau_p, B) ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Panel A: xs_mean bar chart
ax = axes[0]
xs = np.arange(len(df))
colors = ['#4C72B0' if c=='baseline' else '#DD8452' for c in df.cond]
ax.bar(xs, df.xs_mean, color=colors)
ax.set_xticks(xs)
ax.set_xticklabels([f"tau_p={r.tau_p}\nB={r.B}\n{r.cond}" for r in df.itertuples()], fontsize=7)
ax.set_ylabel('Cross-slip mean R')
ax.set_title('Day 95 P3 — Cross-slip mean by (tau_p, B, cond)')
ax.axhline(0.919, color='red', ls='--', lw=0.7, label='Day 94 baseline tie (0.919)')
ax.legend(fontsize=8)

# Panel B: Delta heatmap
ax = axes[1]
Dmat = np.zeros((len(tps), len(Bs)))
for i, tp in enumerate(tps):
    for j, B in enumerate(Bs):
        Dmat[i,j] = dfd[(dfd.tau_p==tp)&(dfd.B==B)]['Delta = R_CNRT - R_base'].values[0]
im = ax.imshow(Dmat, cmap='RdBu_r', vmin=-abs(Dmat).max()-1e-6, vmax=abs(Dmat).max()+1e-6)
ax.set_xticks(range(len(Bs))); ax.set_xticklabels([f"B={b}" for b in Bs])
ax.set_yticks(range(len(tps))); ax.set_yticklabels([f"tau_p={t}" for t in tps])
for i in range(len(tps)):
    for j in range(len(Bs)):
        ax.text(j, i, f"{Dmat[i,j]:+.3f}", ha='center', va='center',
                color='white' if abs(Dmat[i,j])>abs(Dmat).max()*0.6 else 'black', fontsize=10)
ax.set_title(r'Delta = $R^{+CNRT} - R^{base}$')
plt.colorbar(im, ax=ax, fraction=0.05)
plt.tight_layout()
plt.savefig('/tmp/day95_p3_grid.png', dpi=100)
plt.show()


In [5]:

# ---- Best-cell summary ----
best_idx = dfd['Delta = R_CNRT - R_base'].idxmax()
best_row = dfd.iloc[best_idx]
worst_idx = dfd['Delta = R_CNRT - R_base'].idxmin()
worst_row = dfd.iloc[worst_idx]
summary = pd.DataFrame({
    'metric': [
        'best cell',
        'best cell R_+CNRT',
        'best cell R_baseline',
        'best cell Delta',
        'worst cell',
        'worst cell Delta',
        'max positive Delta (Delta_max)',
        'Day 94 P3 Delta_max',
    ],
    'value': [
        f"tau_p={best_row.tau_p}, B={best_row.B}",
        f"{best_row.R_CNRT:.3f}",
        f"{best_row.R_base:.3f}",
        f"{best_row['Delta = R_CNRT - R_base']:+.3f}",
        f"tau_p={worst_row.tau_p}, B={worst_row.B}",
        f"{worst_row['Delta = R_CNRT - R_base']:+.3f}",
        f"{dfd['Delta = R_CNRT - R_base'].max():+.3f}",
        "0.000 (baseline tie only)",
    ]
})
summary


,metric,value
0,best cell,"tau_p=0.01, B=1.0"
1,best cell R_+CNRT,4.817
2,best cell R_baseline,4.880
3,best cell Delta,-0.062
4,worst cell,"tau_p=0.0, B=1.0"
5,worst cell Delta,-0.819
6,max positive Delta (Delta_max),-0.062
7,Day 94 P3 Delta_max,0.000 (baseline tie only)


## 4. 결과 해석

1. **Rehabilitation 성공 여부**: 셀 격자에서 $\Delta_{\max} > 0$ 이면 +CNRT 가 baseline
   을 실제로 초과 → true rehabilitation. Day 94 P3 는 $\Delta_{\max} = 0$ (tie only) 이었다.
2. **Polyak vs replay 의 개별 기여**: $(\tau_p=0.01, B=1)$ 만 켠 셀 vs $(\tau_p=0, B=16)$
   만 켠 셀의 $\Delta$ 비교로 두 처방의 marginal 을 대략 분리할 수 있다. 두 처방이 함께
   켜져야 rehabilitation 이 발생하면 두 처방의 상호작용이 필수.
3. **Generalization gap $\Gamma$**: $\Gamma^{+\text{CNRT}} < \Gamma^{\text{base}}$ 인 셀은
   +CNRT 가 절대 성능은 낮아도 robustness 는 개선. Day 91 P3 의 원래 negative finding 은
   robustness 축에서 반전 가능성.
4. **왜 Polyak / batch 가 도움될 수 있는가**: (i) target Polyak 이 categorical target
   drift 를 완만화 → EMA whitening 초기 warmup pathology 축소, (ii) batch replay 는
   Noisy 헤드의 stochastic gradient 를 평균화해 σ 학습 안정화.

> **결론**: (실행 결과에 따라) Polyak + batch replay 는 Day 94 P3 의 baseline-tie 를 넘어
> 진짜 rehabilitation 을 만들거나, 실패해 +CNRT 열위의 **구조적** 성격을 확증한다. 어느
> 방향이든 §15.29 이후의 처방 설계에 결정적 정보.

**다음 (§15.29) →** (a) 이 rehabilitation 처방을 real-net 으로 옮겨 규모 sensitivity 를
확인, (b) $\Gamma$ 축에서의 robustness 개선이 유지되는지, (c) Cramér vs KL 순위가
Polyak target 안에서 어떻게 재배열되는지.
